# wmh-multisite — WMH-SynthSeg sur les 170 sujets (Kaggle **CPU**), segmentation complète conservée

Étapes 5.4 (méthode M2) et 7.2 (ventricules) de la ROADMAP. **WMH-SynthSeg** (Laso et al., ISBI 2024, FreeSurfer) est un modèle **pré-entraîné généraliste** : il segmente les HSB **et l'anatomie** sur n'importe quel contraste, sans réentraînement.

**Ce que cette version garde (J-045)** : en plus des HSB (label 77, méthode M2), la **segmentation anatomique complète**, dont les **ventricules latéraux** (labels 4 et 43) qui servent à 7.2 (lésions périventriculaires ou profondes). La version précédente ne gardait que le label 77.

**Réglages (J-007, J-043)**
- Mode **CPU d'origine** (le GPU T4 manque de mémoire, J-007) : session **sans accélérateur**, qui ne consomme pas le quota GPU.
- Entrée : la **FLAIR brute** (`orig/FLAIR`), comme le prévoient les auteurs.
- **Les 170 sujets** (60 d'entraînement + 110 de test), en **3 parties équilibrées par scanner** (`PART = "1"`, `"2"`, `"3"` : 57, 57 et 56 sujets, ~7 h chacune, 7,5 min par sujet). `PART = "CHECK"` : les 5 sujets du notebook `06_atrophy_vs_template` (~30 min), pour juger les ventricules avant les 3 grosses parties.
- Un **sous-processus par sujet**, surveillé (RAM, durée) : un sujet trop gourmand est arrêté et noté, la session continue.

**Sorties par sujet** (dans `data/derivatives/wmh-multisite/sub-XXX/anat/`)
| Fichier | Contenu |
|---|---|
| `sub-XXX_space-FLAIR_desc-wmhsynthseg_dseg.nii.gz` | HSB (label 77) sur la grille FLAIR, 0/1 : évaluation de M2 |
| `sub-XXX_space-FLAIR_desc-wmhsynthsegfull_dseg.nii.gz` | segmentation complète (tous les labels) sur la grille FLAIR, plus proche voisin |
| `sub-XXX_space-FLAIR_desc-ventriclessynthseg_mask.nii.gz` | ventricules latéraux (labels 4 et 43) sur la grille FLAIR, 0/1 : 7.2 |
| `sub-XXX_desc-wmhsynthseg1mm_dseg.nii.gz` | sortie brute de l'outil (grille 1 mm), pour contrôle |

## Instructions
1. Régler `PART` dans la cellule de configuration : d'abord `"CHECK"`, puis `"1"`, `"2"`, `"3"` (un notebook ou une version par partie ; elles peuvent tourner en parallèle si Kaggle l'accepte).
2. *Settings* : Accelerator **None** (CPU), Internet **On**. Aucun dataset à ajouter.
3. **Save & Run All (Commit)**.
4. À la fin : onglet **Output**, télécharger `wmhsynthseg_predictions_<PART>.zip`, le décompresser **à la racine du projet**, puis copier le bloc JSON de la dernière cellule dans la conversation.

In [ ]:
# 1. Environment
import json, os, re, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path
import psutil

T0 = time.time()
REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S"),
          "env": {"cpus": os.cpu_count(), "ram_gb": round(psutil.virtual_memory().total / 1e9, 1)}}
print(REPORT["env"])
!pip install -q nibabel 2>&1 | tail -1
import nibabel as nib, numpy as np
from nibabel.processing import resample_from_to
import torch
REPORT["env"]["torch"] = torch.__version__

In [ ]:
PART = "CHECK"      # "CHECK", "1", "2" or "3"

CHECK = ["sub-000", "sub-002", "sub-021", "sub-049", "sub-053"]
ALL_SUBJECTS = {
"sub-000": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/0",
"part": "1"
},
"sub-001": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/1",
"part": "1"
},
"sub-002": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/2",
"part": "2"
},
"sub-003": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/3",
"part": "2"
},
"sub-004": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/4",
"part": "3"
},
"sub-005": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/5",
"part": "3"
},
"sub-006": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/6",
"part": "1"
},
"sub-007": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/7",
"part": "1"
},
"sub-008": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/8",
"part": "2"
},
"sub-009": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/9",
"part": "2"
},
"sub-010": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/10",
"part": "3"
},
"sub-011": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/11",
"part": "3"
},
"sub-012": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/12",
"part": "1"
},
"sub-013": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/13",
"part": "2"
},
"sub-014": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/14",
"part": "3"
},
"sub-015": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/15",
"part": "1"
},
"sub-016": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/16",
"part": "2"
},
"sub-017": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/17",
"part": "1"
},
"sub-018": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/18",
"part": "3"
},
"sub-019": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/19",
"part": "2"
},
"sub-020": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/20",
"part": "1"
},
"sub-021": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/21",
"part": "3"
},
"sub-022": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/22",
"part": "2"
},
"sub-023": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/23",
"part": "1"
},
"sub-024": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/24",
"part": "3"
},
"sub-025": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/25",
"part": "2"
},
"sub-026": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/26",
"part": "1"
},
"sub-027": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/27",
"part": "3"
},
"sub-028": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/28",
"part": "2"
},
"sub-029": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/29",
"part": "1"
},
"sub-030": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/30",
"part": "3"
},
"sub-031": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/31",
"part": "2"
},
"sub-032": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/32",
"part": "1"
},
"sub-033": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/33",
"part": "3"
},
"sub-034": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/34",
"part": "2"
},
"sub-035": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/35",
"part": "1"
},
"sub-036": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/36",
"part": "3"
},
"sub-037": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/37",
"part": "2"
},
"sub-038": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/38",
"part": "1"
},
"sub-039": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/39",
"part": "3"
},
"sub-040": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/40",
"part": "2"
},
"sub-041": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/41",
"part": "1"
},
"sub-042": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/42",
"part": "3"
},
"sub-043": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/43",
"part": "1"
},
"sub-044": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/44",
"part": "2"
},
"sub-045": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/45",
"part": "3"
},
"sub-046": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/46",
"part": "1"
},
"sub-047": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/47",
"part": "2"
},
"sub-048": {
"scanner": "utrecht_philips_3t",
"split": "test",
"dir": "test/Utrecht/48",
"part": "3"
},
"sub-049": {
"scanner": "utrecht_philips_3t",
"split": "training",
"dir": "training/Utrecht/49",
"part": "2"
},
"sub-050": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/50",
"part": "2"
},
"sub-051": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/51",
"part": "3"
},
"sub-052": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/52",
"part": "1"
},
"sub-053": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/53",
"part": "2"
},
"sub-054": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/54",
"part": "3"
},
"sub-055": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/55",
"part": "1"
},
"sub-056": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/56",
"part": "2"
},
"sub-057": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/57",
"part": "3"
},
"sub-058": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/58",
"part": "1"
},
"sub-059": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/59",
"part": "2"
},
"sub-060": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/60",
"part": "3"
},
"sub-061": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/61",
"part": "1"
},
"sub-062": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/62",
"part": "2"
},
"sub-063": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/63",
"part": "3"
},
"sub-064": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/64",
"part": "1"
},
"sub-065": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/65",
"part": "2"
},
"sub-066": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/66",
"part": "3"
},
"sub-067": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/67",
"part": "1"
},
"sub-068": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/68",
"part": "2"
},
"sub-069": {
"scanner": "singapore_siemens_3t",
"split": "training",
"dir": "training/Singapore/69",
"part": "3"
},
"sub-070": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/70",
"part": "2"
},
"sub-071": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/71",
"part": "3"
},
"sub-072": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/72",
"part": "1"
},
"sub-073": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/73",
"part": "2"
},
"sub-074": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/74",
"part": "3"
},
"sub-075": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/75",
"part": "1"
},
"sub-076": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/76",
"part": "2"
},
"sub-077": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/77",
"part": "3"
},
"sub-078": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/78",
"part": "1"
},
"sub-079": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/79",
"part": "2"
},
"sub-080": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/80",
"part": "3"
},
"sub-081": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/81",
"part": "1"
},
"sub-082": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/82",
"part": "2"
},
"sub-083": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/83",
"part": "3"
},
"sub-084": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/84",
"part": "1"
},
"sub-085": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/85",
"part": "2"
},
"sub-086": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/86",
"part": "3"
},
"sub-087": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/87",
"part": "1"
},
"sub-088": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/88",
"part": "2"
},
"sub-089": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/89",
"part": "3"
},
"sub-090": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/90",
"part": "1"
},
"sub-091": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/91",
"part": "2"
},
"sub-092": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/92",
"part": "3"
},
"sub-093": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/93",
"part": "1"
},
"sub-094": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/94",
"part": "2"
},
"sub-095": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/95",
"part": "3"
},
"sub-096": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/96",
"part": "1"
},
"sub-097": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/97",
"part": "2"
},
"sub-098": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/98",
"part": "3"
},
"sub-099": {
"scanner": "singapore_siemens_3t",
"split": "test",
"dir": "test/Singapore/99",
"part": "1"
},
"sub-100": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/100",
"part": "2"
},
"sub-101": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/101",
"part": "3"
},
"sub-102": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/102",
"part": "1"
},
"sub-103": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/103",
"part": "2"
},
"sub-104": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/104",
"part": "3"
},
"sub-105": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/105",
"part": "1"
},
"sub-106": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/106",
"part": "2"
},
"sub-107": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/107",
"part": "3"
},
"sub-108": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/108",
"part": "1"
},
"sub-109": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/109",
"part": "2"
},
"sub-110": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/110",
"part": "3"
},
"sub-111": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/111",
"part": "2"
},
"sub-112": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/112",
"part": "1"
},
"sub-113": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/113",
"part": "2"
},
"sub-114": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/114",
"part": "3"
},
"sub-115": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/115",
"part": "1"
},
"sub-116": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/116",
"part": "2"
},
"sub-117": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/117",
"part": "3"
},
"sub-118": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/118",
"part": "1"
},
"sub-119": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/119",
"part": "2"
},
"sub-120": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/120",
"part": "3"
},
"sub-121": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/121",
"part": "1"
},
"sub-122": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/122",
"part": "2"
},
"sub-123": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/123",
"part": "3"
},
"sub-124": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/124",
"part": "1"
},
"sub-125": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/125",
"part": "2"
},
"sub-126": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/126",
"part": "3"
},
"sub-127": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/127",
"part": "3"
},
"sub-128": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/128",
"part": "1"
},
"sub-129": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/129",
"part": "2"
},
"sub-130": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/130",
"part": "3"
},
"sub-131": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/131",
"part": "1"
},
"sub-132": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/132",
"part": "1"
},
"sub-133": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/133",
"part": "2"
},
"sub-134": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/134",
"part": "3"
},
"sub-135": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/135",
"part": "1"
},
"sub-136": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/136",
"part": "2"
},
"sub-137": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/137",
"part": "2"
},
"sub-138": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/138",
"part": "3"
},
"sub-139": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/139",
"part": "1"
},
"sub-140": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/140",
"part": "2"
},
"sub-141": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/141",
"part": "3"
},
"sub-142": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/142",
"part": "1"
},
"sub-143": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/143",
"part": "2"
},
"sub-144": {
"scanner": "amsterdam_ge_3t",
"split": "training",
"dir": "training/Amsterdam/GE3T/144",
"part": "3"
},
"sub-145": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/145",
"part": "3"
},
"sub-146": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/146",
"part": "1"
},
"sub-147": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/147",
"part": "2"
},
"sub-148": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/148",
"part": "3"
},
"sub-149": {
"scanner": "amsterdam_ge_3t",
"split": "test",
"dir": "test/Amsterdam/GE3T/149",
"part": "1"
},
"sub-150": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/150",
"part": "1"
},
"sub-151": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/151",
"part": "2"
},
"sub-152": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/152",
"part": "3"
},
"sub-153": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/153",
"part": "1"
},
"sub-154": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/154",
"part": "2"
},
"sub-155": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/155",
"part": "3"
},
"sub-156": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/156",
"part": "1"
},
"sub-157": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/157",
"part": "2"
},
"sub-158": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/158",
"part": "3"
},
"sub-159": {
"scanner": "amsterdam_ge_1p5t",
"split": "test",
"dir": "test/Amsterdam/GE1T5/159",
"part": "1"
},
"sub-160": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./160",
"part": "1"
},
"sub-161": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./161",
"part": "2"
},
"sub-162": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./162",
"part": "3"
},
"sub-163": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./163",
"part": "1"
},
"sub-164": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./164",
"part": "2"
},
"sub-165": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./165",
"part": "3"
},
"sub-166": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./166",
"part": "1"
},
"sub-167": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./167",
"part": "2"
},
"sub-168": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./168",
"part": "3"
},
"sub-169": {
"scanner": "amsterdam_philips_3t",
"split": "test",
"dir": "test/Amsterdam/Philips_VU .PETMR_01./169",
"part": "1"
}
}
SUBJECTS = {s: v for s, v in ALL_SUBJECTS.items() if (s in CHECK if PART == "CHECK" else v["part"] == PART)}
print(PART, len(SUBJECTS), "subjects")

In [ ]:
# 2. Raw FLAIR (and manual reference, for a quick check) from DataverseNL
API = "https://dataverse.nl/api/datasets/:persistentId/?persistentId=doi:10.34894/AECRSD"
FILES = json.load(urllib.request.urlopen(API, timeout=60))["data"]["latestVersion"]["files"]
by_path = {f"{f.get('directoryLabel') or ''}/{f['dataFile']['filename']}": f["dataFile"]["id"] for f in FILES}
IN = Path("/tmp/wmhss_in"); IN.mkdir(exist_ok=True)
t0 = time.time()
for sub, info in SUBJECTS.items():
    for rel, name in [("orig/FLAIR.nii.gz", f"{sub}_FLAIR.nii.gz"), ("wmh.nii.gz", f"{sub}_wmh.nii.gz")]:
        out = IN / name
        if not out.exists():
            urllib.request.urlretrieve(f"https://dataverse.nl/api/access/datafile/{by_path[info['dir'] + '/' + rel]}", out)
REPORT["download_s"] = round(time.time() - t0, 1)
print("downloaded in", REPORT["download_s"], "s")

In [ ]:
# 3. WMH-SynthSeg code (FreeSurfer dev branch, GitHub) + model (790 MB, MGH FTP), standalone
WS = Path("/tmp/wmhss"); (WS / "unet3d").mkdir(parents=True, exist_ok=True)
FS = Path("/tmp/fakefs/models"); FS.mkdir(parents=True, exist_ok=True)
B = "https://raw.githubusercontent.com/freesurfer/freesurfer/dev/mri_WMHsynthseg/WMHSynthSeg"
for f in ["inference.py", "utils.py"]:
    urllib.request.urlretrieve(f"{B}/{f}", WS / f)
for f in ["__init__.py", "buildingblocks.py", "model.py", "se.py", "utils.py"]:
    urllib.request.urlretrieve(f"{B}/unet3d/{f}", WS / "unet3d" / f)
MODEL = FS / "WMH-SynthSeg_v10_231110.pth"
if not MODEL.exists():
    urllib.request.urlretrieve("https://ftp.nmr.mgh.harvard.edu/pub/dist/lcnpublic/dist/WMH-SynthSeg/WMH-SynthSeg_v10_231110.pth", MODEL)
# Launcher: allow the numpy types stored in the checkpoint (PyTorch >= 2.6 loads with weights_only=True)
(WS / "run_wmhss.py").write_text("""
import runpy, sys, numpy as np, torch
core = getattr(np, "_core", None) or np.core
if hasattr(torch.serialization, "add_safe_globals"):
    torch.serialization.add_safe_globals([(core.multiarray.scalar, "numpy.core.multiarray.scalar"), np.dtype]
        + [getattr(np.dtypes, n) for n in ("Float64DType", "Float32DType", "Int64DType") if hasattr(np, "dtypes")])
sys.argv = ["inference.py"] + sys.argv[1:]
runpy.run_path("inference.py", run_name="__main__")
""")
ENV = dict(os.environ, FREESURFER_HOME="/tmp/fakefs")
# Labels of the lateral ventricles in the model's label list (checked in inference.py: 4 = left, 43 = right)
src = (WS / "inference.py").read_text()
assert "'left-lateral-ventricle'" in src and "'right-lateral-ventricle'" in src, "label list changed upstream"
print("model", round(MODEL.stat().st_size / 1e6), "MB")

In [ ]:
# 4. One guarded subprocess per subject; outputs: WMH (77), full segmentation, lateral ventricles (4, 43)
RAM_LIMIT_GB = round(0.95 * psutil.virtual_memory().total / 1e9, 1)
WMH, VENTRICLES = 77, (4, 43)
OUT = Path("/kaggle/working/data/derivatives/wmh-multisite"); RAW_OUT = Path("/tmp/wmhss_out"); RAW_OUT.mkdir(exist_ok=True)

def run_one(sub):
    flair = IN / f"{sub}_FLAIR.nii.gz"; seg = RAW_OUT / f"{sub}_seg.nii.gz"
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, "run_wmhss.py", "--i", str(flair), "--o", str(seg), "--device", "cpu",
                             "--crop", "--threads", "-1"], cwd=WS, env=ENV, stdout=open(RAW_OUT / f"{sub}.log", "w"),
                            stderr=subprocess.STDOUT)
    pp, peak, killed = psutil.Process(proc.pid), 0.0, None
    while proc.poll() is None:
        try:
            rss = sum(c.memory_info().rss for c in [pp] + pp.children(recursive=True)) / 1e9
        except psutil.Error:
            rss = 0
        peak = max(peak, rss)
        if rss > RAM_LIMIT_GB or time.time() - t0 > 1800:
            killed = f"RAM {rss:.1f} GB" if rss > RAM_LIMIT_GB else "timeout 30 min"
            for c in pp.children(recursive=True) + [pp]:
                try: c.kill()
                except psutil.Error: pass
            break
        time.sleep(1)
    row = {"participant_id": sub, "scanner": SUBJECTS[sub]["scanner"], "split": SUBJECTS[sub]["split"],
           "seconds": round(time.time() - t0, 1), "peak_ram_gb": round(peak, 1), "killed": killed,
           "ok": seg.exists() and killed is None}
    if row["ok"]:
        ref = nib.load(flair)
        anat = OUT / sub / "anat"; anat.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(seg, anat / f"{sub}_desc-wmhsynthseg1mm_dseg.nii.gz")                 # raw output, 1 mm grid
        full = np.rint(resample_from_to(nib.load(seg), ref, order=0).get_fdata()).astype(np.uint8)  # labels <= 85
        save = lambda data, name: nib.save(nib.Nifti1Image(data.astype(np.uint8), ref.affine), anat / name)
        save(full, f"{sub}_space-FLAIR_desc-wmhsynthsegfull_dseg.nii.gz")
        pred = full == WMH
        save(pred, f"{sub}_space-FLAIR_desc-wmhsynthseg_dseg.nii.gz")
        vent = np.isin(full, VENTRICLES)
        save(vent, f"{sub}_space-FLAIR_desc-ventriclessynthseg_mask.nii.gz")
        gt = np.rint(nib.load(IN / f"{sub}_wmh.nii.gz").get_fdata()); keep = gt != 2; g = gt == 1; p = pred & keep
        ml = float(np.prod(ref.header.get_zooms()[:3])) / 1000
        row |= {"dice_quick": round(float(2 * (p & g).sum() / max(p.sum() + g.sum(), 1)), 3),
                "true_ml": round(float(g.sum() * ml), 2), "pred_ml": round(float(p.sum() * ml), 2),
                "ventricles_ml": round(float(vent.sum() * ml), 1)}
    return row

REPORT["subjects"] = []
for i, sub in enumerate(SUBJECTS, 1):
    row = run_one(sub)
    REPORT["subjects"].append(row)
    print(f"[{(time.time() - T0) / 3600:.2f} h] {i}/{len(SUBJECTS)} {sub} {row['scanner']:22s} ok={row['ok']} "
          f"{row['seconds']:.0f} s RAM {row['peak_ram_gb']} GB dice={row.get('dice_quick')} "
          f"vent={row.get('ventricles_ml')} ml killed={row['killed']}", flush=True)

In [ ]:
# 5. Archive (BIDS-named files, to unzip at the repository root) + summary JSON
archive = Path(f"/kaggle/working/wmhsynthseg_predictions_{PART}.zip")
with zipfile.ZipFile(archive, "w") as z:
    for f in sorted(OUT.rglob("*.nii.gz")):
        z.write(f, f.relative_to("/kaggle/working"))
import pandas as pd
df = pd.DataFrame(REPORT["subjects"])
df.to_csv(f"/kaggle/working/wmhsynthseg_runs_{PART}.csv", index=False)
ok = df[df.ok]
REPORT["summary"] = {"ok": int(df.ok.sum()), "failed": df[~df.ok].participant_id.tolist(),
                     "seconds_median": float(df.seconds.median()), "peak_ram_max_gb": float(df.peak_ram_gb.max()),
                     "dice_quick_by_scanner": ok.groupby("scanner").dice_quick.mean().round(3).to_dict() if len(ok) else {},
                     "ventricles_ml_median_by_scanner": ok.groupby("scanner").ventricles_ml.median().round(1).to_dict() if len(ok) else {}}
if PART == "CHECK":
    REPORT["summary"]["check_subjects"] = ok.set_index("participant_id")[["dice_quick", "ventricles_ml"]].to_dict("index")
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S"); REPORT["session_h"] = round((time.time() - T0) / 3600, 2)
REPORT["part"] = PART
json.dump(REPORT, open(f"/kaggle/working/wmhsynthseg_report_{PART}.json", "w"), indent=1)
print(archive, round(archive.stat().st_size / 1e6, 1), "MB")
print("\n=== Copier ce bloc dans la conversation ===")
print(json.dumps({k: v for k, v in REPORT.items() if k != "subjects"}, indent=1))